## Import libaries

In [7]:
import torch
import mne
import numpy as np

from braindecode.models import ShallowFBCSPNet
from braindecode.util import set_random_seeds

from pathlib import Path
import sys

current_dir = Path().resolve().parent.parent
sys.path.append(str(current_dir))

# Own implementations
from src.utilities.preprocessing import Filtering, EEG_preprocessing, EMG_preprocessing #E402
from src.utilities.trainer_and_evaluator import FusionNet_train_eval, SingleNet_train_eval
from src.utilities.load_and_visualize_data import load_datasets, visualize_EEG

In [8]:
# Set random seed to ensure reproducible initialization below
seed = 20240205
cuda = torch.cuda.is_available()
set_random_seeds(seed=seed, cuda=cuda)

## Functions

In [25]:
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def bandpass_filter(data, fs, low, high, order=4):
    """
    Zero-phase Butterworth bandpass filter
    data: (samples, channels)
    """
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, data, axis=0)


def standardize_epochs(X):
    """
    Channel-wise z-score normalization
    X shape: (trials, samples, channels)
    """
    Xn = np.zeros_like(X)
    for ch in range(X.shape[2]):
        scaler = StandardScaler()
        Xn[:, :, ch] = scaler.fit_transform(X[:, :, ch])
    return Xn

#=============#
# CORE LOADER #
#=============#
def load_subject(subject_id, sel_mode, sel_class_type):
    '''
    Parameters
    ----------
    sel_class_type : str
        '769' for left hand MI
        '770' for right hand MI
    '''
    event_dict = {
        '769' : 'left_hand',
        '770' : 'right_hand',
        '783' : 'Cue_unknown'
    }

    FS = 250  # Sampling frequency
    TMIN = 0.0              # start at cue
    TMAX = 6.0              # 4 seconds MI
    ONSET_TMIN = 0.0        # onset window
    ONSET_TMAX = 3.0        # End of onset
    BANDPASS = (8, 32)  # mu + beta band

    mode = "T" if sel_mode else "E"
    fname = f"A{subject_id:02d}{mode}.gdf"

    filepath = Path().resolve().parent / 'utilities/BCI2a_dataset' / fname

    # Load GDF file
    raw = mne.io.read_raw_gdf(str(filepath), preload=True, verbose=False)

    # Drop EOG channels
    eeg_chs = raw.info["ch_names"][:22]
    raw.pick(eeg_chs)

    # Extract events
    events, event_id = mne.events_from_annotations(raw)
    
    print(event_id)
    class_type = event_id[sel_class_type]

    # keep only MI events
    events_mi = events[events[:, 2] == class_type]

    # Epoching
    epochs = mne.Epochs(
        raw,
        events_mi,
        event_id={event_dict[sel_class_type]: class_type},
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False
    )

    # crop to only onset
    epochs_onset = epochs.copy().crop(tmin=ONSET_TMIN, tmax = ONSET_TMAX)

    X = epochs_onset.get_data()           # (trials, channels, samples)
    X = np.transpose(X, (0, 2, 1))  # Reorder to (trials, samples, channels)

    continous_datastream = epochs.get_data()                                # (trials, channels, samples)
    continous_datastream = np.transpose(continous_datastream, (0, 2, 1))    # Reorder to (trials, samples, channels)

    # Labels (single source of truth)
    y = np.array([events_mi[e] for e in epochs.events[:, 2]])


    for i in range(X.shape[0]):     # Move this above epochs
        X[i] = bandpass_filter(X[i], FS, BANDPASS[0], BANDPASS[1])

    # Normalize
    X = standardize_epochs(X)

    print(X.shape)           # (n_trials, time, 22)
    print(np.unique(y))      # [0 1 2 3]
    assert len(X) == len(y)

    return X, y, continous_datastream

# ---------------------------
# LOAD ALL SUBJECTS
# ---------------------------

def load_all_subjects(sel_mode=True, sel_class_type = '0', num_subj = 2):
    """
    Loads all 9 subjects
    """
    if num_subj < 2 or num_subj > 10:
        raise ValueError(f'num_subj {num_subj} must be between 2 and 10')
    X_all, y_all = [], []

    for subj in range(1, num_subj):
        X, y, cds = load_subject(subject_id = subj, sel_mode = sel_mode, sel_class_type = sel_class_type)

        X_all.append(X)
        y_all.append(y)

        print(f"Subject {subj}: {X.shape}, labels: {np.unique(y)}")

    return np.concatenate(X_all), np.concatenate(y_all)

class SingleManageDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels):
        '''
        Takes in the concatinated dataset of all trials, samples and channels.
        Args:
            X [ndArray] - with the dimension of (trials, samples, channels)
            y [int] - Indicate the number of trials 
        '''
        # Convert to tensors
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.rng = np.random.default_rng(seed)

        print('data shape:', self.data.shape)
        print('labels shape:', self.labels.shape)
    
    def create_test_train_dataset(self, train_procent = 0.8):
        '''
        Return:
            train_dataset - 
            test_dataset -
        '''
        N_labels = len(self.labels)
        indices = self.rng.permutation(N_labels)            # Shuffle labels
        
        train_size = int(train_procent * N_labels)
        train_idx = indices[:train_size]
        test_idx  = indices[train_size:]

        return train_idx, test_idx
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

    def get_rng_generator(self):
        return self.rng

class pre():
    def __init__(self,
                 fs = 125,
                 bandpass_lowcut : int = 8,
                 bandpass_highcut : int = 30,
                 trial_period : int = 9,
                 trim_period : int = 3):
        
        self.fs = fs
        self.lowcut = bandpass_lowcut
        self.highcut = bandpass_highcut
        self.trial_period = trial_period
        self.trim_period = trim_period
        self.expected_num_epochs = 30

        
    def preprocessing_routine(self,
                            raw_eeg : np.ndarray) -> tuple[np.ndarray, int]:
        
        '''
        Performs the full preprocessing routine:
        1) Notch + Bandpass filter
        2) Resample + z-score standardization + Secmentation into epochs

        Parameters
        ----------
        raw_eeg : np.ndarray
            This holds keys for a specfic class (finger). NOTE - If raw_eeg is a list, it will be converted to a dict with key 'single_class'. 2D array - Dim(samples, channels)
        bandpass_lowcut : int
            Lowpass frequency
        bandpass_highcut : int
            Highpass frequency
        all_markers : list
            A log of all marker data including zero
        extract_event : str
            Choise which segment of data to extract (For example: 'ALL', 'CONTRACT', 'RELEASE', 'REST')
        
        Return
        ------
        :return: np.ndarray of normalized EEG data
        :return: Int of the total amount of epochs for one experiment
        '''

        # ---------------------------#
        # 1) NOTCH + BANDPASS FILTER #
        # ---------------------------#
        EEG_filter_ins = Filtering(fs = self.fs)
        
        EEG_notch = EEG_filter_ins.notch(raw_eeg, cutoff=50, Q=30)
        EEG_bandpass, _ = EEG_filter_ins.butter_bandpass(EEG_notch, lowcut = self.lowcut, highcut = self.highcut, order=4)

        #===============================#
        # 2) Calculate number of epochs #
        #===============================#
        num_trim_samples = self.fs * self.trim_period * 2                   # Total samples from trim period. WHY *2 : Trim egde on both sides
        num_valid_samples = EEG_bandpass.shape[0] - num_trim_samples        # Total samples for experimental period
        cal_num_epochs = num_valid_samples / (self.fs * self.trial_period)  # Divide out total samples in sections of trial periods -> Results in number of epochs
        if cal_num_epochs != self.expected_num_epochs:                      # Inform if epochs is differnet from usual amount. Can happen if bad trials is removed.
            print(f'OBSERVATION - NUM OF EPOCH ({cal_num_epochs}) IS DIFFERNET FROM USUAL {self.expected_num_epochs}')
        
        cal_num_epochs = int(np.round(cal_num_epochs))


        # --------#
        # 3) TRIM #
        # --------#
        trim_idx_102 = self.fs * self.trim_period
        trim_idx_201 = (self.fs * self.trial_period * cal_num_epochs) + trim_idx_102        # WHY instead of data[trim : -trim] -> Inconsistency in protocol causes the last batch of data not be included -> Rare but can happen
        EEG_trim = EEG_bandpass[trim_idx_102 : trim_idx_201, :]
        print(f"Original shape {EEG_bandpass.shape}\n"
              f"Trim 102 idx {trim_idx_102}\n"
              f"Trim 201 idx {trim_idx_201}\n"
              f'EEG_trim shape: {EEG_trim.shape}\n')
        

        #EEG_car = EEG_trim - np.mean(EEG_trim, axis = 1, keepdims = True)       # (S, C)

        # power = np.abs(EEG_car) ** 2
        # from scipy.signal import hilbert
        # power = np.abs(hilbert(EEG_car, axis=0)) ** 2

        ## TRY TO REMOVE NOISE IN THE REST PERIOD
        ## SEE HOW THE POWER CHANGES

        # --------------------------------------------------
        # 3) Z-SCORE STANDARDIZATION
        # --------------------------------------------------
        EEG_norm = EEG_filter_ins.zscore(EEG_trim, mode = 'within_ch')
        
        return EEG_norm, cal_num_epochs

## Load datasets

In [37]:
# X_train_left, y_train_left = load_all_subjects(sel_mode=True, sel_class_type='769', num_subj=10)
# X_train_right, y_train_right = load_all_subjects(sel_mode=True, sel_class_type='770', num_subj=10)
import pandas as pd
''' LOAD MARKERS FOR PLOTS
marker_index_files01 = load_ins.find_flex_files(
    subjects = ["subject_0", "subject_1"],
    modality = 'Markers',
    fingers = 'index',
    prefix = 'flex'
)

marker_thumb_files01 = load_ins.find_flex_files(
    subjects = ["subject_0", "subject_1"],
    modality = 'Markers',
    fingers = 'thumb',
    prefix = 'flex'
)

marker_index_files2 = load_ins.find_flex_files(
    subjects = ["subject_2"],
    modality = 'Markers',
    fingers = 'index',
    prefix = 'flex'
)

marker_thumb_files2 = load_ins.find_flex_files(
    subjects = ["subject_2"],
    modality = 'Markers',
    fingers = 'thumb',
    prefix = 'flex'
)

markers_index01 = load_ins.load_datasets_marker(
    path_to_data_files = marker_index_files01
)

markers_thumb01 = load_ins.load_datasets_marker(
    path_to_data_files = marker_thumb_files01
)

markers_index2 = load_ins.load_datasets_marker(
    path_to_data_files = marker_index_files2
)

markers_thumb2 = load_ins.load_datasets_marker(
    path_to_data_files = marker_thumb_files2
)'''

'''INDEX VS. THUMB
EEG_index_files01 = load_ins.find_flex_files(
    subjects=["subject_0", "subject_1"],
    modality="EEG",
    fingers=["index"],
    prefix = 'flex'
)

EEG_thumb_files01 = load_ins.find_flex_files(
    subjects=["subject_0", "subject_1"],
    modality="EEG",
    fingers=["thumb"],
    prefix = 'flex'
)

EEG_index_files2 = load_ins.find_flex_files(
    subjects=["subject_2"],
    modality="EEG",
    fingers=["index"],
    prefix = 'flex'
)

EEG_thumb_files2 = load_ins.find_flex_files(
    subjects=["subject_2"],
    modality="EEG",
    fingers=["thumb"],
    prefix = 'flex'
)


EEG_ins01 = EEG_preprocessing(fs = EEG_FREQ)
EEG_ins2 = EEG_preprocessing(fs = EEG_FREQ)

EEG_index01, EEG_index_epoch01, EEG_index_epoch_mean01, total_epochs = load_ins.load_datasets_EEG(
    path_to_data_files = EEG_index_files01,
    preprocessing_func = EEG_ins01.preprocessing_routine,
    bandpass_lowcut = EEG_LOWCUT,
    bandpass_highcut = EEG_HIGHCUT,
    extract_event = 'contract',
    trial_period = 4
    )

EEG_thumb01, EEG_thumb_epoch01, EEG_thumb_epoch_mean01, total_epochs = load_ins.load_datasets_EEG(
    path_to_data_files = EEG_thumb_files01,
    preprocessing_func = EEG_ins01.preprocessing_routine,
    bandpass_lowcut = EEG_LOWCUT,
    bandpass_highcut = EEG_HIGHCUT,
    extract_event = 'contract',
    trial_period = 4)

EEG_index2, EEG_index_epoch2, EEG_index_epoch_mean2, total_epochs = load_ins.load_datasets_EEG(
    path_to_data_files = EEG_index_files2,
    preprocessing_func = EEG_ins2.preprocessing_routine,
    bandpass_lowcut = EEG_LOWCUT,
    bandpass_highcut = EEG_HIGHCUT,
    extract_event = 'contract',
    trial_period = 3
    )

EEG_thumb2, EEG_thumb_epoch2, EEG_thumb_epoch_mean2, total_epochs = load_ins.load_datasets_EEG(
    path_to_data_files = EEG_thumb_files2,
    preprocessing_func = EEG_ins2.preprocessing_routine,
    bandpass_lowcut = EEG_LOWCUT,
    bandpass_highcut = EEG_HIGHCUT,
    extract_event = 'contract',
    trial_period = 3
    )'''

base_dir = Path().resolve().parent / 'experiment/data'

load_ins = load_datasets(base_dir = base_dir)

EEG_ins = pre(fs = 125,
              bandpass_lowcut = 0.5,
              bandpass_highcut = 30,
              trial_period = 9,
              trim_period = 3)

EEG_index_files = load_ins.find_flex_files(
        subjects = ['subject_0_closedEyes', 'subject_1_closedEyes'],
        modality = 'EEG',
        fingers = 'index',
        prefix = 'flex'
    )

EEG_thumb_files = load_ins.find_flex_files(
        subjects = ['subject_0_closedEyes', 'subject_1_closedEyes'],
        modality = 'EEG',
        fingers = 'thumb',
        prefix = 'flex'
    )

total_epochs = 0
eeg_num_ch = 16
all_data = []


for data_file in EEG_index_files:

    EEG_df = pd.read_csv(data_file)
    EEG_raw = EEG_df.iloc[:, 1:17].to_numpy()
    EEG_marker_log = EEG_df.iloc[:, -1].to_numpy()

    EEG_filt, num_epochs = EEG_ins.preprocessing_routine(raw_eeg = EEG_raw)
    
    all_data.append(EEG_filt)
    total_epochs += num_epochs
                          # Only load one data

EEG = np.concatenate(all_data, axis = 0)
print(f"Reshaped data shape: {EEG.shape}")

EEG_samples_per_epoch = EEG.shape[0] // total_epochs      
print(f"Samples per epoch: {EEG_samples_per_epoch}")

EEG_epoch_index = EEG.reshape(total_epochs, EEG_samples_per_epoch, eeg_num_ch)

EEG_epoch_mean = EEG_epoch_index.mean(axis=0)
print(f"Epoched data shape: {EEG_epoch_index.shape}")
print(f"Mean epoch data shape: {EEG_epoch_mean.shape}")


total_epochs = 0
eeg_num_ch = 16
all_data = []

for data_file in EEG_thumb_files:

    EEG_df = pd.read_csv(data_file)
    EEG_raw = EEG_df.iloc[:, 1:17].to_numpy()
    EEG_marker_log = EEG_df.iloc[:, -1].to_numpy()

    EEG_filt, num_epochs = EEG_ins.preprocessing_routine(raw_eeg = EEG_raw)
    
    all_data.append(EEG_filt)
    total_epochs += num_epochs
                          # Only load one data

EEG = np.concatenate(all_data, axis = 0)
print(f"Reshaped data shape: {EEG.shape}")

EEG_samples_per_epoch = EEG.shape[0] // total_epochs      
print(f"Samples per epoch: {EEG_samples_per_epoch}")

EEG_epoch_thumb = EEG.reshape(total_epochs, EEG_samples_per_epoch, eeg_num_ch)

EEG_epoch_mean = EEG_epoch_thumb.mean(axis=0)
print(f"Epoched data shape: {EEG_epoch_thumb.shape}")
print(f"Mean epoch data shape: {EEG_epoch_mean.shape}")

OBSERVATION - NUM OF EPOCH (30.061333333333334) IS DIFFERNET FROM USUAL 30
Original shape (34569, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (30.008) IS DIFFERNET FROM USUAL 30
Original shape (34509, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (30.008) IS DIFFERNET FROM USUAL 30
Original shape (34509, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (30.008) IS DIFFERNET FROM USUAL 30
Original shape (34509, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (30.00888888888889) IS DIFFERNET FROM USUAL 30
Original shape (34510, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (30.061333333333334) IS DIFFERNET FROM USUAL 30
Original shape (34569, 16)
Trim 102 idx 375
Trim 201 idx 34125
EEG_trim shape: (33750, 16)

OBSERVATION - NUM OF EPOCH (0.99022

In [38]:
from torch.utils.data import Subset
from torch.utils.data import DataLoader

X_cl1 = EEG_epoch_index[:, 3*125 : 6*125, :]
X_cl2 = EEG_epoch_thumb[:, 3*125 : 6*125, :]

# BCI 2a
X, labels = None, None
X = np.concatenate(( X_cl1,
                     X_cl2)
                   )

X = X.transpose(0, 2, 1)

labels = np.concatenate( (np.zeros(X_cl1.shape[0]), 
                          np.ones(X_cl2.shape[0]))
                          )
print('X shape : ', X.shape)
print('labels shape :', labels.shape)


X shape :  (361, 16, 375)
labels shape : (361,)


## Classfier

In [39]:
from skorch.dataset import ValidSplit
from braindecode import EEGClassifier
import torch.nn as nn

model = ShallowFBCSPNet(
    n_chans=16,
    n_times=1000,
    n_outputs=2,
    final_conv_length="auto",
)

net = EEGClassifier(
    'ShallowFBCSPNet',
    criterion = nn.CrossEntropyLoss(),
    lr = 1e-3,
    train_split=ValidSplit(0.2),
    max_epochs = 150,
    batch_size = 60,
    #iterator_train = train_loader,
    #iterator_valid = test_loader
    # To train a neural network you need validation split, here, we use 20%.
)

In [31]:
from braindecode.datautil import infer_signal_properties

sig_props = infer_signal_properties(X, labels, mode="classification")
print(f"Inferred signal properties:\n{sig_props}")

Inferred signal properties:
{'n_outputs': 2, 'n_times': 375, 'n_chans': 16}


In [40]:
net.fit(X, labels)

  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.9441       0.5479        1.4501  0.3273
      2        0.8941       0.5479        1.1597  0.2554
      3        0.9121       0.5479        0.9649  0.2664
      4        0.8675       0.5479        0.8762  0.2616
      5        0.8613       0.5205        0.8157  0.2605
      6        0.7921       0.5479        0.7693  0.2576
      7        0.7599       0.5753        0.7426  0.3654
      8        0.7822       0.5479        0.7290  0.2777
      9        0.7930       0.5616        0.7194  0.2495
     10        0.8173       0.5753        0.7151  0.2649
     11        0.7419       0.5753        0.7134  0.2544
     12        0.7752       0.5890        0.7120  0.2504
     13        0.7346       0.6027        0.7089  0.2674
     14        0.7275       0.6027        0.7084  0.2608
     15        0.7012       0.5890        0.7090  0.2465
     16        0.7320       0.5

<class 'braindecode.classifier.EEGClassifier'>[initialized](
  module_==================================================================================================================================================
  Layer (type (var_name):depth-idx)             Input Shape               Output Shape              Param #                   Kernel Shape
  =================================================================================================================================================
  ShallowFBCSPNet (ShallowFBCSPNet)             [1, 16, 375]              [1, 2]                    --                        --
  ├─Ensure4d (ensuredims): 1-1                  [1, 16, 375]              [1, 16, 375, 1]           --                        --
  ├─Rearrange (dimshuffle): 1-2                 [1, 16, 375, 1]           [1, 1, 375, 16]           --                        --
  ├─CombinedConv (conv_time_spat): 1-3          [1, 1, 375, 16]           [1, 40, 351, 1]           26,640                    --
  ├─BatchNorm2d (bnorm): 1-4                    [1, 40, 351, 1]           [1, 40, 351, 1]           80                        --
  ├─Expression (conv_nonlin_exp): 1-5           [1, 40, 351, 1]           [1, 40, 351, 1]           --                        --
  ├─AvgPool2d (pool): 1-6                       [1, 40, 351, 1]           [1, 40, 19, 1]            --                        [75, 1]
  ├─SafeLog (pool_nonlin_exp): 1-7              [1, 40, 19, 1]            [1, 40, 19, 1]            --                        --
  ├─Dropout (drop): 1-8                         [1, 40, 19, 1]            [1, 40, 19, 1]            --                        --
  ├─Sequential (final_layer): 1-9               [1, 40, 19, 1]            [1, 2]                    --                        --
  │    └─Conv2d (conv_classifier): 2-1          [1, 40, 19, 1]            [1, 2, 1, 1]              1,522                     [19, 1]
  │    └─SqueezeFinalOutput (squeeze): 2-2      [1, 2, 1, 1]              [1, 2]                    --                        --
  │    │    └─Rearrange (squeeze): 3-1          [1, 2, 1, 1]              [1, 2, 1]                 --                        --
  =================================================================================================================================================
  Total params: 28,242
  Trainable params: 28,242
  Non-trainable params: 0
  Total mult-adds (Units.MEGABYTES): 0.00
  =================================================================================================================================================
  Input size (MB): 0.02
  Forward/backward pass size (MB): 0.11
  Params size (MB): 0.01
  Estimated Total Size (MB): 0.14
  =================================================================================================================================================,
)